# Model Experiments: Kryptonite-20, -30 and -40

The paper claims that polynomial logistic regression stays below 59% up to degree 10, that more data does not help, and that no model reaches the 90% target. It also asks which other models work and what degree and how many examples a polynomial would need. This notebook answers these in order, with one protocol, scores the final pipeline on a locked test set and writes the hidden-set predictions to `hiddenlabels/` at the repository root. The helper code is in `experiment_tools.py`. Finished results are also kept in `exp_cache/` (git-ignored, not needed to reproduce anything) so that an interrupted run resumes; delete that folder for a fresh run.

Set the environment variable `KRYP_QUICK=1` for a dry run on 3,000 rows with small grids; its numbers mean nothing and no predictions are written.

In [1]:
import experiment_tools as et
from experiment_tools import G, NS, cv_by_fold, cv_config

print(
    "QUICK dry run" if et.QUICK else "Full run",
    f"- {et.K}-fold CV on 80% of the rows, 20% locked test",
)

Full run - 5-fold CV on 80% of the rows, 20% locked test


## Data and Protocol

A stratified 20% of the labelled rows is a locked test set, read only at the end. The other 80% is split into five stratified folds. Every comparison and hyper-parameter choice uses these folds, and anything learned from the data, including which bits to use, is learned on the training folds only. Accuracies are reported as mean ± standard deviation over the five folds, and two models are compared by their difference on each fold and how many folds each wins. The locked-test accuracy gets the binomial interval p ± 1.96·√(p(1−p)/N). The inputs are ±1 bits plus noise of standard deviation about 0.1, so `sign(X)` recovers the bits.

In [2]:
et.check_protocol()

n=20: 12000 labelled rows = 9600 dev (5 folds of 7680 train / 1920 validation rows) + 2400 locked; 0.496 positive
n=30: 20000 labelled rows = 16000 dev (5 folds of 12800 train / 3200 validation rows) + 4000 locked; 0.499 positive
n=40: 32000 labelled rows = 25600 dev (5 folds of 20480 train / 5120 validation rows) + 6400 locked; 0.498 positive


## Experiment 1: Higher Polynomial Degree on All Bits

*Claim:* polynomial logistic regression never exceeds 59%, even at degree 10. *Hypothesis:* a higher degree does not fix this on all bits, because the number of monomials grows far faster than the number of training rows.

Logistic regression on explicit monomials (products of distinct bits) up to the largest degree that fits in memory (4, 3, 2 for n = 20, 30, 40), with L1 or L2 and `C` tuned by CV. Beyond that, the paper's own polynomial kernel trained in the dual at degree 6, 10, 15, n and 2n. Degree n can represent every function of n bits, so the model class is not the limit.

In [3]:
et.test_kernel()
for n in NS:
    cv_config(n, "BASE", "paper", "all", "deg2", dict(deg=2, C=0.85))
    for d in range(1, G["MAX_DEG"][n] + 1):
        for pen in ("l2", "l1"):
            for C in G["C_L2" if pen == "l2" else "C_L1"]:
                cv_config(n, "LR", f"lr_{pen}", "all", f"deg{d}", dict(deg=d, C=C))
    for d in G["KP_DEGS"][n]:
        cv_config(n, "KPOLY", "kpoly", "all", f"deg{d}", dict(deg=d, C=G["KP_C"]))
et.table_degree_all()

kernel tests passed
BASE n=20 deg2 degree 2, C 0.85                                  0.544 ± 0.010  (14.0s)
LR n=20 deg3 degree 3, C 0.003                                   0.563 ± 0.014  (5.1s)
LR n=20 deg3 degree 3, C 0.01                                    0.572 ± 0.016  (5.3s)
LR n=20 deg3 degree 3, C 0.03                                    0.574 ± 0.007  (6.7s)
LR n=20 deg3 degree 3, C 0.1                                     0.552 ± 0.005  (8.8s)
LR n=20 deg4 degree 4, C 0.001                                   0.581 ± 0.004  (5.8s)
LR n=20 deg4 degree 4, C 0.01                                    0.570 ± 0.005  (6.4s)
LR n=20 deg4 degree 4, C 0.1                                     0.567 ± 0.007  (7.2s)
LR n=20 deg4 degree 4, C 1                                       0.567 ± 0.007  (6.5s)
LR n=20 deg4 degree 4, C 0.003                                   0.621 ± 0.013  (30.1s)
LR n=20 deg4 degree 4, C 0.01                                    0.675 ± 0.017  (31.7s)
LR n=20 deg4 degree 

## Experiment 2: Other Models on All Bits

*Claim:* no model reaches 90%; the paper suggests neural networks trained from scratch, tree-based and clustering-based models as future work. Each is tuned by the same CV on all n bits: K-means where each cluster predicts its majority training label, a random forest, and an MLP with KryptoNet's shape (four hidden layers of 512) trained from scratch with Adam and early stopping. The MLP takes about an hour on a laptop CPU; each fold is stored when it finishes.

In [ ]:
for n in NS:
    for k in G["KM_K"]:
        cv_config(n, "KM", "kmeans", "all", "kmeans", dict(k=k))
    for f in G["RF_MAXF"]:
        for leaf in G["RF_LEAF"]:
            cv_config(n, "RF", "rf", "all", "rf", dict(max_features=f, msl=leaf))
    if et.RUN_MLP:
        cv_by_fold(n, "mlp", dict(h=G["MLP_H"]), "4x512")
et.table_families()

KM n=20 kmeans k 128                                             0.504 ± 0.020  (5.5s)
RF n=20 rf max features sqrt, min leaf 1                         0.632 ± 0.012  (7.6s)
RF n=20 rf max features sqrt, min leaf 5                         0.616 ± 0.007  (6.5s)
RF n=20 rf max features 0.5, min leaf 1                          0.714 ± 0.015  (6.6s)
RF n=20 rf max features 0.5, min leaf 5                          0.703 ± 0.010  (6.6s)
RF n=20 rf max features 1.0, min leaf 1                          0.822 ± 0.021  (7.0s)
RF n=20 rf max features 1.0, min leaf 5                          0.824 ± 0.023  (6.4s)
FOLD n=20 4x512, fold 1 layers 4×512                             0.932 ± 0.000  (135.2s)
FOLD n=20 4x512, fold 2 layers 4×512                             0.930 ± 0.000  (177.3s)
FOLD n=20 4x512, fold 3 layers 4×512                             0.920 ± 0.000  (83.1s)
FOLD n=20 4x512, fold 4 layers 4×512                             0.940 ± 0.000  (137.9s)
FOLD n=20 4x512, fold 5 layers 4×512

## Experiment 3: Which Bits Does the Forest Use?

The forest on all bits does better than the polynomials, and a forest only splits on bits that help, so we ask it which bits it relies on. We use recursive feature elimination (Guyon et al., 2002), a standard method that uses nothing about how the data were made:

1. Fit a random forest on the bits still in and record its out-of-bag (OOB) accuracy: each tree is scored on the training rows it did not see, so no held-out rows are needed.
2. Remove the bit with the lowest impurity importance (how much the forest's splits on that bit reduce the training error).
3. Repeat until one bit is left.

From the n sets on this path we keep the smallest one whose OOB accuracy is within one standard error, √(p(1−p)/N), of the best set with at most 14 bits (the one-standard-error rule: prefer the simpler set unless the larger one is clearly better). The 14-bit cap keeps every polynomial on the kept bits in memory. All of this runs on the training rows of each fold, so the held-out fold plays no part.

*Hypothesis:* if only some bits carry the label, the OOB accuracy rises as irrelevant bits are removed and drops sharply when the first relevant bit goes. The figure shows the path of every fold.

In [ ]:
et.table_elimination()
et.fig_elimination()

## Experiment 4: Polynomial Degree on the Selected Bits

*Future-work question:* what degree does a polynomial need? The same model class as Experiment 1, logistic regression (L2) on all monomials up to degree d, now on the bits kept inside each training fold. Ten bits have at most 1,023 monomials, so every degree up to 10 fits.

In [ ]:
for n in NS:
    for d in G["SEL_DEGS"]:
        for C in G["C_L2"]:
            cv_config(n, "LR", "lr_l2", "sel", f"deg{d}", dict(deg=d, C=C))
et.table_degree_sel()
et.fig_degree()

## Experiment 5: Amount of Training Data

*Claim:* quadrupling the training data does not help. *Future-work question:* how many examples are needed? The best model of each kind is trained on nested random subsets of the training folds and scored on the held-out fold (first three folds). For LR, `C` is scaled by (full rows / rows used) to keep the penalty per row fixed.

In [ ]:
et.fig_learning_curves(et.run_learning_curves())

## Final Pipeline and Hidden-Set Predictions

The final pipeline for each n is the candidate with the highest CV mean, the paper baseline excluded. It is refitted on all dev rows and scored on the locked 20%. Two controls on the first three folds should both give about 0.5: shuffled training labels (otherwise something leaks), and the same model on as many eliminated bits (otherwise the kept bits are not what carries the label). The pipeline is then refitted on all labelled rows and applied to the hidden inputs; the code checks the predictions, writes `hiddenlabels/y_predicted_{20,30,40}.npy` and runs the course's `check_submission.py`. `y_predicted_40_adapt.npy` comes from the LoRA notebook.

In [ ]:
FINAL = et.choose_final()
et.run_controls(FINAL)
et.table_final(FINAL)
print()
et.predict_hidden(FINAL)

## Summary

One row per approach. All rows are CV means except the last.

In [ ]:
et.fig_summary_table(FINAL)
et.table_summary(FINAL)

## Compute Used

Total model-fitting time recorded in `exp_cache/`, for the environmental impact assessment.

In [ ]:
CPU_WATTS = 80
GRID_G_PER_KWH = 143.96  # figures from the report
stores = (et.CV, et.LC, et.SEL, et.FIN)
hours = sum(r.get("secs", 0) for st in stores for r in st.rows.values()) / 3600
print(
    f"{hours:.2f} h of model fitting, about {CPU_WATTS * hours / 1000:.2f} kWh and"
    f" {CPU_WATTS * hours * GRID_G_PER_KWH / 1000:.0f} g CO2e"
)